# 3. Saving and Loading

TidalPy can persist a world or a system in two ways:

- **TOML config** (`save_to_toml`): a human-readable recipe. It stores the identity, bulk properties, and layer structure, and rebuilds the object by re-running the builder.
- **Binary snapshot** (`save_binary` / `load_binary`): a compressed serialization of the object's state, restored without re-running the builder.

This notebook round-trips a world and a system through both formats and checks what each preserves.


In [4]:
import tempfile
import timeit
from pathlib import Path

from TidalPy.structures_x.configs import build_world, build_system, load_toml
from TidalPy.structures_x.worlds.layered import LayeredWorld
from TidalPy.structures_x.system import System

work_dir = Path(tempfile.mkdtemp(prefix="tidalpy_demo_"))
print("writing files to:", work_dir)


writing files to: C:\Users\joepr\AppData\Local\Temp\tidalpy_demo_7cs93da5


## Saving a World as TOML

`save_to_toml` writes the config recipe. Reading it back with `build_world` reconstructs the world.

In [8]:
earth = build_world("earth_simple")

toml_path = work_dir / "earth.toml"
earth.save_to_toml(str(toml_path))
print(toml_path.read_text())

earth_from_toml = build_world(load_toml(str(toml_path)))
print("rebuilt:", earth_from_toml.name, "| layers:", earth_from_toml.num_layers)
print()

print("Time to build a world from a toml:")
%timeit build_world(load_toml(str(toml_path)))

schema_version = "0.2.0"
name = "Earth-Simple"
type = "terrestrial"
radius_m = 6371000.0
mass_kg = 5.972e+24
spin_frequency_rad_s = 7.292e-5

[layers.core]
class = "physics"
type = "iron"
layer_index = 0
radius_outer_m = 3480000.0
is_tidal = false

[layers.mantle]
class = "solidliquid"
type = "mantle_rock"
layer_index = 1
radius_fraction = 1.0
is_tidal = true

rebuilt: Earth-Simple | layers: 2

202 μs ± 7.81 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


## Saving a World as a Binary Snapshot

`save_binary` writes the exact state. To read it back, load into an existing world instance. Any placeholder will do: its state is replaced by the file's contents.

In [10]:
binary_path = work_dir / "earth.tpyb"
earth.save_binary(str(binary_path))
print("binary size:", binary_path.stat().st_size, "bytes")

restored = LayeredWorld("placeholder", 1.0, 1.0)   # placeholder state is overwritten on load
restored.load_binary(str(binary_path))
print("restored:", restored.name, "| R:", restored.radius, "| M:", restored.mass,
      "| layers:", restored.num_layers)
print()

print("Time to load a world from binary:")
%timeit restored.load_binary(str(binary_path))

binary size: 1349 bytes
restored: Earth-Simple | R: 6371000.0 | M: 5.972e+24 | layers: 2

Time to load a world from binary:
38.3 μs ± 1.34 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


## Inspecting a World's Live State

`get_config_dict` returns the world's current state as a dictionary: identity, bulk properties, and every layer. The live-state dictionary is a report, not a builder recipe. To rebuild a world in memory, load the TOML recipe saved above (as in demo 1) or use the binary snapshot.

In [4]:
state = earth.get_config_dict()
print("state keys:", list(state))
print("layers    :", list(state["layers"]))

# Rebuild in memory from the portable TOML recipe (a plain dict), the same path demo 1 used.
recipe = load_toml(str(toml_path))
clone = build_world(recipe)
print("clone:", clone.name, "| layers:", clone.num_layers, "| mean density:", round(clone.calc_mean_density(), 1))


state keys: ['schema_version', 'name', 'type', 'radius_m', 'mass_kg', 'albedo', 'emissivity', 'obliquity_rad', 'spin_frequency_rad_s', 'tides', 'layers']
layers    : ['core', 'mantle']
clone: Earth-Simple | layers: 2 | mean density: 5513.3


## Saving and Loading Systems

A whole system (a host, a star, and orbiting worlds with their orbital elements) supports the same two formats. The binary snapshot restores each world as its concrete type and keeps the orbital elements and roles, so derived quantities such as insolation come back unchanged.


In [11]:
system = build_system("sol_system")

sys_toml = work_dir / "sol_system.toml"
sys_binary = work_dir / "sol_system.tpyb"
system.save_to_toml(str(sys_toml))
system.save_binary(str(sys_binary))
print("system toml  :", sys_toml.stat().st_size, "bytes")
print("system binary:", sys_binary.stat().st_size, "bytes")

loaded = System()
loaded.load_binary(str(sys_binary))
print("name         :", loaded.name)
print("worlds       :", [(w.name, type(w).__name__) for w in loaded])
print("host / star  :", loaded.host.name, "/", loaded.star.name)
print("Earth insolation preserved:",
      round(loaded.calc_insolation_flux('earth'), 2), "W/m^2")


system toml  : 466 bytes
system binary: 1983 bytes
name         : Sol System
worlds       : [('sun', 'StarWorld'), ('earth', 'LayeredWorld'), ('jupiter', 'GasGiantWorld')]
host / star  : sun / sun
Earth insolation preserved: 1361.35 W/m^2


## Choosing a Format

- Use TOML when a human needs to read or edit the definition, when it belongs in version control, or when sharing a world with someone else.
- Use binary to snapshot and restore an object's exact state, for example to cache a built system between runs. About 10x faster than rebuilding from a TOML.

A binary file carries every layer and its attached physics models, including each layer's equation-of-state model, but not derived results or the world-level tide and spin models. After loading, re-run the equation-of-state solve and reattach any tide or spin model, as when the object is first built.


In [12]:
import shutil
shutil.rmtree(work_dir, ignore_errors=True)   # clean up the temporary files
print("cleaned up", work_dir)


cleaned up C:\Users\joepr\AppData\Local\Temp\tidalpy_demo_7cs93da5
